In [1]:
!pip install requests pandas streamlit

In [2]:
import requests
import pandas as pd
import sqlite3
import json
from datetime import datetime

In [3]:
API_HOST = "aerodatabox.p.rapidapi.com"

API_KEY = "5288304d14msh75e8f3c3e5b24dbp1dbb58jsn8ce4b72688da"

HEADERS = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": API_HOST
}

print("API Configuration Completed")

API Configuration Completed


In [4]:
url = "https://aerodatabox.p.rapidapi.com/airports/iata/DEL"

response = requests.get(url, headers=HEADERS)

print("Status Code:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("Airport:", data.get("name"))
    print("IATA:", data.get("iata"))
    print("ICAO:", data.get("icao"))
else:
    print(response.text)

Status Code: 200
Airport: None
IATA: DEL
ICAO: VIDP


In [5]:
conn = sqlite3.connect("airtracker.db")
cursor = conn.cursor()

print("Database Created Successfully")

Database Created Successfully


In [6]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS airport (
    airport_id INTEGER PRIMARY KEY AUTOINCREMENT,
    iata_code TEXT UNIQUE,
    icao_code TEXT,
    airport_name TEXT,
    city TEXT,
    country TEXT
)
""")

conn.commit()

print("Airport Table Created Successfully")

Airport Table Created Successfully


In [7]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS aircraft (
    aircraft_id INTEGER PRIMARY KEY AUTOINCREMENT,
    registration TEXT UNIQUE,
    model TEXT,
    manufacturer TEXT,
    icao_type_code TEXT,
    owner TEXT
)
""")

conn.commit()

print("Aircraft Table Created Successfully")

Aircraft Table Created Successfully


In [8]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS flights (
    flight_id INTEGER PRIMARY KEY AUTOINCREMENT,
    flight_number TEXT,
    airline_code TEXT,
    origin_iata TEXT,
    destination_iata TEXT,
    status TEXT,
    aircraft_code TEXT
)
""")

conn.commit()

print("Flights Table Created Successfully")

Flights Table Created Successfully


In [9]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS airport_delays (
    delay_id INTEGER PRIMARY KEY AUTOINCREMENT,
    airport_iata TEXT,
    delay_date TEXT,
    average_delay INTEGER,
    arrival_delay INTEGER,
    departure_delay INTEGER
)
""")

conn.commit()

print("Airport Delays Table Created Successfully")

Airport Delays Table Created Successfully


In [10]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

[('airport',), ('sqlite_sequence',), ('aircraft',), ('flights',), ('airport_delays',)]


In [11]:
airports = [
    ("DEL", "VIDP", "Delhi Airport", "Delhi", "India"),
    ("BOM", "VABB", "Mumbai Airport", "Mumbai", "India"),
    ("BLR", "VOBL", "Bengaluru Airport", "Bengaluru", "India"),
    ("HYD", "VOHS", "Hyderabad Airport", "Hyderabad", "India"),
    ("MAA", "VOMM", "Chennai Airport", "Chennai", "India"),
    ("CCU", "VECC", "Kolkata Airport", "Kolkata", "India"),
    ("GOI", "VOGO", "Goa Airport", "Goa", "India"),
    ("COK", "VOCI", "Kochi Airport", "Kochi", "India"),
    ("AMD", "VAAH", "Ahmedabad Airport", "Ahmedabad", "India"),
    ("PNQ", "VAPO", "Pune Airport", "Pune", "India"),
    ("DXB", "OMDB", "Dubai Airport", "Dubai", "UAE"),
    ("SIN", "WSSS", "Singapore Changi", "Singapore", "Singapore"),
    ("LHR", "EGLL", "London Heathrow", "London", "UK"),
    ("JFK", "KJFK", "John F. Kennedy", "New York", "USA"),
    ("DOH", "OTHH", "Hamad International", "Doha", "Qatar")
]

cursor.executemany("""
INSERT OR IGNORE INTO airport
(iata_code, icao_code, airport_name, city, country)
VALUES (?, ?, ?, ?, ?)
""", airports)

conn.commit()

print("15 Airports Inserted Successfully")

15 Airports Inserted Successfully


In [12]:
cursor.execute("SELECT COUNT(*) FROM airport")
print("Total Airports:", cursor.fetchone()[0])

Total Airports: 15


In [13]:
url = "https://aerodatabox.p.rapidapi.com/flights/airports/iata/DEL"

response = requests.get(url, headers=HEADERS)

print("Status Code:", response.status_code)

data = response.json()

print(type(data))

# Show first few keys returned by API
if isinstance(data, dict):
    print(data.keys())

Status Code: 200
<class 'dict'>
dict_keys(['departures', 'arrivals'])


In [14]:
print("Number of Departures:", len(data["departures"]))

if len(data["departures"]) > 0:
    print(data["departures"][0])

Number of Departures: 382
{'movement': {'airport': {'icao': 'VAUD', 'iata': 'UDR', 'name': 'Udaipur', 'countryCode': 'in', 'timeZone': 'Asia/Kolkata'}, 'scheduledTime': {'utc': '2026-06-04 06:29Z', 'local': '2026-06-04 11:59+05:30'}, 'revisedTime': {'utc': '2026-06-04 06:29Z', 'local': '2026-06-04 11:59+05:30'}, 'terminal': '2', 'gate': 'T34', 'quality': ['Basic', 'Live']}, 'number': 'AI 1737', 'status': 'Departed', 'codeshareStatus': 'IsOperator', 'isCargo': False, 'aircraft': {'model': 'Airbus A320 NEO'}, 'airline': {'name': 'Air India', 'iata': 'AI', 'icao': 'AIC'}}


In [15]:
flight = data["departures"][0]

print("Keys:")
print(flight.keys())

Keys:
dict_keys(['movement', 'number', 'status', 'codeshareStatus', 'isCargo', 'aircraft', 'airline'])


In [16]:
import json

print(json.dumps(data["departures"][0], indent=2)[:3000])

{
  "movement": {
    "airport": {
      "icao": "VAUD",
      "iata": "UDR",
      "name": "Udaipur",
      "countryCode": "in",
      "timeZone": "Asia/Kolkata"
    },
    "scheduledTime": {
      "utc": "2026-06-04 06:29Z",
      "local": "2026-06-04 11:59+05:30"
    },
    "revisedTime": {
      "utc": "2026-06-04 06:29Z",
      "local": "2026-06-04 11:59+05:30"
    },
    "terminal": "2",
    "gate": "T34",
    "quality": [
      "Basic",
      "Live"
    ]
  },
  "number": "AI 1737",
  "status": "Departed",
  "codeshareStatus": "IsOperator",
  "isCargo": false,
  "aircraft": {
    "model": "Airbus A320 NEO"
  },
  "airline": {
    "name": "Air India",
    "iata": "AI",
    "icao": "AIC"
  }
}


In [17]:
departures = data["departures"][:100]

for flight in departures:

    flight_number = flight.get("number", "")

    airline_code = flight.get("airline", {}).get("iata", "")

    destination = flight.get("movement", {}).get("airport", {}).get("name", "")

    status = flight.get("status", "")

    cursor.execute("""
    INSERT INTO flights
    (flight_number, airline_code, origin_iata, destination_iata, status, aircraft_code)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    (
        flight_number,
        airline_code,
        "DEL",
        destination,
        status,
        "UNKNOWN"
    ))

conn.commit()

print("Flights Inserted Successfully")

Flights Inserted Successfully


In [18]:
cursor.execute("SELECT COUNT(*) FROM flights")
print("Total Flights:", cursor.fetchone()[0])

Total Flights: 400


In [19]:
cursor.execute("""
SELECT flight_number,
       airline_code,
       origin_iata,
       destination_iata,
       status
FROM flights
LIMIT 10
""")

for row in cursor.fetchall():
    print(row)

('AI 917', 'AI', 'DEL', 'Dubai', 'Canceled')
('AI 2927', 'AI', 'DEL', 'Mumbai', 'Departed')
('AI 1792', 'AI', 'DEL', 'Srinagar', 'Departed')
('6E 6211', '6E', 'DEL', 'Agartala', 'Departed')
('6E 291', '6E', 'DEL', 'Aizawl', 'Departed')
('BA 256', 'BA', 'DEL', 'London', 'GateClosed')
('6E 2132', '6E', 'DEL', 'Pune', 'Departed')
('6E 2701', '6E', 'DEL', 'Ayodhya', 'Departed')
('AI 2995', 'AI', 'DEL', 'Mumbai', 'GateClosed')
('6E 2298', '6E', 'DEL', 'Jodhpur', 'Departed')


In [20]:
cursor.execute("""
SELECT airline_code,
       COUNT(*) as total_flights
FROM flights
GROUP BY airline_code
ORDER BY total_flights DESC
LIMIT 10
""")

for row in cursor.fetchall():
    print(row)

('6E', 163)
('AI', 126)
('IX', 39)
('SG', 18)
('QP', 10)
('', 8)
('TG', 4)
('KC', 4)
('T5', 3)
('MH', 3)


In [21]:
cursor.execute("""
SELECT status,
       COUNT(*) as total
FROM flights
GROUP BY status
ORDER BY total DESC
""")

flight_status = cursor.fetchall()

for row in flight_status:
    print(row)

('Departed', 257)
('GateClosed', 99)
('Boarding', 16)
('Expected', 14)
('Canceled', 6)
('CheckIn', 5)
('Delayed', 3)


In [22]:
cursor.execute("""
SELECT origin_iata || ' -> ' || destination_iata AS route,
       COUNT(*) as total
FROM flights
GROUP BY route
ORDER BY total DESC
LIMIT 10
""")

top_routes = cursor.fetchall()

for row in top_routes:
    print(row)

('DEL -> Mumbai', 30)
('DEL -> Srinagar', 24)
('DEL -> Leh', 14)
('DEL -> Lucknow', 13)
('DEL -> Hyderabad', 13)
('DEL -> Bangalore', 13)
('DEL -> London', 12)
('DEL -> Dibrugarh', 12)
('DEL -> Siliguri', 11)
('DEL -> Ahmedabad', 11)


In [23]:
cursor.execute("""
SELECT COUNT(*)
FROM aircraft
""")

print("Aircraft Records:", cursor.fetchone()[0])

cursor.execute("""
SELECT registration, model
FROM aircraft
LIMIT 10
""")

for row in cursor.fetchall():
    print(row)

Aircraft Records: 5
('VT-ANB', 'Airbus A320')
('VT-SCG', 'Boeing 737-800')
('VT-EXA', 'Airbus A321')
('VT-JKP', 'Boeing 777-300ER')
('A6-EON', 'Airbus A380')


In [24]:
cursor.execute("PRAGMA table_info(flights)")

for row in cursor.fetchall():
    print(row)

(0, 'flight_id', 'INTEGER', 0, None, 1)
(1, 'flight_number', 'TEXT', 0, None, 0)
(2, 'airline_code', 'TEXT', 0, None, 0)
(3, 'origin_iata', 'TEXT', 0, None, 0)
(4, 'destination_iata', 'TEXT', 0, None, 0)
(5, 'status', 'TEXT', 0, None, 0)
(6, 'aircraft_code', 'TEXT', 0, None, 0)


In [25]:
cursor.execute("""
SELECT DISTINCT aircraft_code
FROM flights
LIMIT 20
""")

for row in cursor.fetchall():
    print(row)

('UNKNOWN',)


In [26]:
aircraft_data = [
    ("VT-ANB", "Airbus A320", "Airbus", "A320", "Air India"),
    ("VT-SCG", "Boeing 737-800", "Boeing", "B738", "SpiceJet"),
    ("VT-EXA", "Airbus A321", "Airbus", "A321", "IndiGo"),
    ("VT-JKP", "Boeing 777-300ER", "Boeing", "B77W", "Vistara"),
    ("A6-EON", "Airbus A380", "Airbus", "A388", "Emirates")
]

cursor.executemany("""
INSERT OR IGNORE INTO aircraft
(registration, model, manufacturer, icao_type_code, owner)
VALUES (?, ?, ?, ?, ?)
""", aircraft_data)

conn.commit()

print("Aircraft Data Inserted Successfully")

Aircraft Data Inserted Successfully


In [27]:
cursor.execute("SELECT * FROM aircraft")

for row in cursor.fetchall():
    print(row)

(1, 'VT-ANB', 'Airbus A320', 'Airbus', 'A320', 'Air India')
(2, 'VT-SCG', 'Boeing 737-800', 'Boeing', 'B738', 'SpiceJet')
(3, 'VT-EXA', 'Airbus A321', 'Airbus', 'A321', 'IndiGo')
(4, 'VT-JKP', 'Boeing 777-300ER', 'Boeing', 'B77W', 'Vistara')
(5, 'A6-EON', 'Airbus A380', 'Airbus', 'A388', 'Emirates')


In [28]:
cursor.execute("PRAGMA table_info(airport_delays)")

for row in cursor.fetchall():
    print(row)

(0, 'delay_id', 'INTEGER', 0, None, 1)
(1, 'airport_iata', 'TEXT', 0, None, 0)
(2, 'delay_date', 'TEXT', 0, None, 0)
(3, 'average_delay', 'INTEGER', 0, None, 0)
(4, 'arrival_delay', 'INTEGER', 0, None, 0)
(5, 'departure_delay', 'INTEGER', 0, None, 0)


In [29]:
delay_data = [
    ("DEL", "2025-06-01", 120, 15, 18),
    ("BOM", "2025-06-01", 110, 12, 20),
    ("BLR", "2025-06-01", 95, 10, 18),
    ("HYD", "2025-06-01", 90, 8, 16),
    ("MAA", "2025-06-01", 85, 6, 14)
]

cursor.executemany("""
INSERT INTO airport_delays
(
airport_iata,
delay_date,
average_delay,
arrival_delay,
departure_delay
)
VALUES (?, ?, ?, ?, ?)
""", delay_data)

conn.commit()

print("Airport Delay Data Inserted Successfully")

Airport Delay Data Inserted Successfully


In [30]:
cursor.execute("SELECT * FROM airport_delays")

for row in cursor.fetchall():
    print(row)

(1, 'DEL', '2025-06-01', 120, 15, 18)
(2, 'BOM', '2025-06-01', 110, 12, 20)
(3, 'BLR', '2025-06-01', 95, 10, 18)
(4, 'HYD', '2025-06-01', 90, 8, 16)
(5, 'MAA', '2025-06-01', 85, 6, 14)
(6, 'DEL', '2025-06-01', 120, 15, 18)
(7, 'BOM', '2025-06-01', 110, 12, 20)
(8, 'BLR', '2025-06-01', 95, 10, 18)
(9, 'HYD', '2025-06-01', 90, 8, 16)
(10, 'MAA', '2025-06-01', 85, 6, 14)
(11, 'DEL', '2025-06-01', 120, 15, 18)
(12, 'BOM', '2025-06-01', 110, 12, 20)
(13, 'BLR', '2025-06-01', 95, 10, 18)
(14, 'HYD', '2025-06-01', 90, 8, 16)
(15, 'MAA', '2025-06-01', 85, 6, 14)
(16, 'DEL', '2025-06-01', 120, 15, 18)
(17, 'BOM', '2025-06-01', 110, 12, 20)
(18, 'BLR', '2025-06-01', 95, 10, 18)
(19, 'HYD', '2025-06-01', 90, 8, 16)
(20, 'MAA', '2025-06-01', 85, 6, 14)


In [31]:
cursor.execute("SELECT COUNT(*) FROM airport")
print("Airports:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM aircraft")
print("Aircraft:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM flights")
print("Flights:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM airport_delays")
print("Airport Delays:", cursor.fetchone()[0])

Airports: 15
Aircraft: 5
Flights: 400
Airport Delays: 20


In [32]:
app_code = '''
import streamlit as st
import sqlite3
import pandas as pd

st.set_page_config(
    page_title="Air Tracker Dashboard",
    page_icon="✈️",
    layout="wide"
)

conn = sqlite3.connect("flight.db")
cursor = conn.cursor()

st.title("✈️ Air Tracker Dashboard")

# Dashboard Metrics
col1, col2, col3, col4 = st.columns(4)

cursor.execute("SELECT COUNT(*) FROM airport")
airports = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM aircraft")
aircraft = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM flights")
flights = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM airport_delays")
delays = cursor.fetchone()[0]

col1.metric("Airports", airports)
col2.metric("Aircraft", aircraft)
col3.metric("Flights", flights)
col4.metric("Delay Records", delays)

st.divider()
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created successfully")

app.py created successfully


In [33]:
more_code = '''

st.subheader("🔍 Flight Search")

flight_no = st.text_input("Enter Flight Number")

if flight_no:
    query = """
    SELECT flight_number,
           airline_code,
           origin_iata,
           destination_iata,
           status
    FROM flights
    WHERE flight_number = ?
    """

    df = pd.read_sql_query(
        query,
        conn,
        params=(flight_no,)
    )

    st.dataframe(df)

st.divider()

st.subheader("🛫 Airport Information")

airport_df = pd.read_sql_query(
    "SELECT * FROM airport",
    conn
)

st.dataframe(airport_df)

st.divider()

st.subheader("📊 Flight Status Analysis")

status_df = pd.read_sql_query("""
SELECT status,
       COUNT(*) as total
FROM flights
GROUP BY status
""", conn)

st.bar_chart(
    status_df.set_index("status")
)

'''
with open("app.py", "a") as f:
    f.write(more_code)

print("Flight Search Added")

Flight Search Added


In [34]:
final_code = '''

st.divider()

st.subheader("🏆 Top Routes")

routes_df = pd.read_sql_query("""
SELECT origin_iata || ' → ' || destination_iata AS route,
       COUNT(*) AS total
FROM flights
GROUP BY route
ORDER BY total DESC
LIMIT 10
""", conn)

st.dataframe(routes_df)

st.divider()

st.subheader("⏱ Airport Delay Analysis")

delay_df = pd.read_sql_query("""
SELECT airport_iata,
       average_delay,
       arrival_delay,
       departure_delay
FROM airport_delays
""", conn)

st.dataframe(delay_df)

st.bar_chart(
    delay_df.set_index("airport_iata")[["average_delay"]]
)

conn.close()
'''

with open("app.py", "a") as f:
    f.write(final_code)

print("Dashboard Completed Successfully")

Dashboard Completed Successfully


In [35]:
import os
print(os.listdir())

['.config', 'flight.db', 'app.py', 'airtracker.db', 'sample_data']


In [36]:
import sqlite3

conn = sqlite3.connect("flight.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

[]


In [37]:
import sqlite3

conn = sqlite3.connect("airtracker.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

[('airport',), ('sqlite_sequence',), ('aircraft',), ('flights',), ('airport_delays',)]


In [38]:
with open("app.py", "r") as f:
    content = f.read()

content = content.replace(
    'sqlite3.connect("flight.db")',
    'sqlite3.connect("airtracker.db")'
)

with open("app.py", "w") as f:
    f.write(content)

print("app.py updated successfully")

app.py updated successfully


In [39]:
with open("app.py", "r") as f:
    content = f.read()

print("airtracker.db" in content)

True


In [45]:
!pkill -f streamlit
!pip uninstall -y streamlit
!pip install streamlit==1.35.0 -q

Found existing installation: streamlit 1.35.0
Uninstalling streamlit-1.35.0:
  Successfully uninstalled streamlit-1.35.0


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.42.96.100:8501

your url is: https://swift-streets-tie.loca.lt
y
